<a href="https://colab.research.google.com/github/SoupBoi1/TLS-Fingerprinting-Malicious-Detection/blob/main/TLSMaliciousDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#TLSFingerprintingMaliciousDetectionNotebook1

#author Matt Lutjen, Brady Bangasser, Sudipta Halder,

#Importing data and libs

In [ ]:
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import requests

In [ ]:


#preprocessing
from sklearn.preprocessing import normalize
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import StandardScaler

#imbalance
from imblearn.datasets import make_imbalance
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

#datasets
from sklearn.datasets import load_iris
from imblearn.datasets import make_imbalance
from sklearn.datasets import make_classification
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin

from sklearn.model_selection import train_test_split

#imbalanced learning
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

#models
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neural_network import MLPClassifier
from sklearn.svm import OneClassSVM

#matrics
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve, auc
from sklearn.metrics import average_precision_score
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import confusion_matrix # imported:  it outputes fp,fn,tp,tn



In [ ]:
import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization,Dense, Conv2D, Flatten, Reshape
from tensorflow.keras.layers import Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input

In [ ]:
# todo add the data set plz
url = 'https://ja4db.com/api/read/'

response = requests.get(url)
data = response.json()
#df_web = pd.DataFrame(data)

In [7]:
df = pd.read_csv('https://raw.githubusercontent.com/SoupBoi1/TLS-Fingerprinting-Malicious-Detection/refs/heads/main/dataset/dataset.csv')

In [8]:

display(df.head())

,JA4Shash,SNI,JA3hash,Version,JA3Shash,OrgName,DstPort,AppName,DstIP,Type,SrcIP,SrcPort,JA4hash,Attack
0,t120400_c02f_460f64128655,officehymy.com,a0e9f5d64349fb13191bc781f81f42e1,0.0,61be9ce3d068c08ff99a857f62352f9d,Japan Network Information Center; XSERVER20 (JP),443,sodinokibi,157.112.183.48,M,10.127.0.109,58688,t12d190800_d83cc789557e_7af1ed941c26,1
1,t1206h1_c02c_e1dda4771ae8,sf16-muse-va.ibytedtos.com,6ec2896feff5746955f700c0023f5804,0.0,42ec7b1db61428bf1cc6e01b9ef02b04,Slovak Telecom Network Administrator; BA-WEBHO...,443,star_quiz_01,212.5.219.10,M,192.168.0.100,51187,t12d1409h1_c866b44c5a26_b39be8c56a14,1
2,t1304h2_1302_a56c5b993250,avatars.mds.yandex.net,cd08e31494f9531f560d64c695473da9,0.0,4ee87de303b9c8138441e6527cbeea3e,YANDEX LLC; YANDEX-87-250-247 (RU),443,zloader,87.250.247.182,M,10.127.0.108,61625,t13d1516h2_8daaf6152771_e5627efa2ab1,1
3,t1304h2_1301_a56c5b993250,js.stripe.com,579ccef312d18482fc42e2b822ca2430,0.0,60c22edca21e0598ad28f8d78c8dedcd,Amazon.com; Inc.; AMAZON-CF,443,asyncrat,18.245.86.52,M,10.127.0.20,50972,t13d1715h2_5b57614c22b0_a815a0c236aa,1
4,t120400_c014_cbb8871a0652,rps-svcs.oracle.com,2a458dd9c65afbcf591cd8c2a194b804,0.0,a860078aa51e1102b117d1f8a1437b72,Akamai International; BV; AIBV,443,bazarbackdoor,23.222.50.60,M,10.127.0.3,50579,t12d210600_b973bfd88a0e_1da50ec048a3,1


In [9]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30664 entries, 0 to 30663
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   JA4Shash  28951 non-null  object 
 1   SNI       30239 non-null  object 
 2   JA3hash   30664 non-null  object 
 3   Version   28951 non-null  float64
 4   JA3Shash  28951 non-null  object 
 5   OrgName   30664 non-null  object 
 6   DstPort   30664 non-null  int64  
 7   AppName   30664 non-null  object 
 8   DstIP     30664 non-null  object 
 9   Type      30414 non-null  object 
 10  SrcIP     30664 non-null  object 
 11  SrcPort   30664 non-null  int64  
 12  JA4hash   30664 non-null  object 
 13  Attack    30664 non-null  int64  
dtypes: float64(1), int64(3), object(10)
memory usage: 3.3+ MB


,Version,DstPort,SrcPort,Attack
count,28951.0,30664.000000,30664.000000,30664.000000
mean,0.0,583.606933,55442.169482,0.787666
std,0.0,1485.352244,4529.980500,0.408966
min,0.0,80.000000,33139.000000,0.000000
25%,0.0,443.000000,51291.750000,1.000000
50%,0.0,443.000000,54972.500000,1.000000
75%,0.0,443.000000,58615.250000,1.000000
max,0.0,49816.000000,65533.000000,1.000000


#Pre-possessing and Data Analysis

## 60/20/20 split


In [ ]:
X_train,X_val, X_test = #todo
y_train,y_val, y_test = #todo

#Model Cration, Training, Validation,Testing

#Results

#Model Export